# Чекпоинт 7: Наблюдаемость через MLflow — обучение, регистрация и анализ

## 1. Введение и цель

Данный ноутбук реализует пункт 9 задания:

1. Подключение к MLflow Tracking Server и MinIO (S3-совместимое хранилище).
2. Обоснование выбора архитектуры Meta-Labeling как финальной модели проекта.
3. Подготовка данных по той же логике, что и в `src/train.py::_prepare_data`.
4. Переобучение Meta-Labeling с полным логированием гиперпараметров, метрик и артефактов.
5. Регистрация модели в MLflow Model Registry с тегом/alias `PRD`.
6. Анализ ошибок: разбор ложных срабатываний и пропусков на тест-выборке.
7. Сравнение с baseline-стратегиями (Buy&Hold, логистическая регрессия).
8. Проверка устойчивости (robustness) к малым возмущениям входных данных.
9. Фиксация воспроизводимости: seed, версии библиотек, описание данных.
10. Выводы по чекпоинту.

## 2. Подготовка окружения и подключение к MLflow

In [ ]:
# Демо-режим: уменьшает число эпох для быстрого запуска без GPU/сервера
DEMO_MODE = True
DEMO_EPOCHS = 3  # в продакшен-запуске используйте epochs=30

In [ ]:
import os
import sys
import ast as _ast
import warnings
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings("ignore")

# Добавляем корень проекта в sys.path, чтобы видеть пакет src
sys.path.insert(0, str(Path.cwd().parent))

from src.common import (
    set_seed,
    load_splits,
    resolve_y_column,
    forward_return,
    triple_barrier_labels,
    evaluate_on_forward,
    sharpe_ratio,
    make_synthetic_splits,
    SEED,
    ACTIVE_ASSET,
    HORIZON,
    MIN_HOLDING,
    COST_BPS,
    VOL_WINDOW_TB,
    PT_MULT,
    SL_MULT,
    PER_YEAR_FWD,
)
from src.model import MetaLabelingModel, MetaLabelingPyfunc

print("Импорты выполнены успешно.")
print(f"  ACTIVE_ASSET={ACTIVE_ASSET}, HORIZON={HORIZON}, SEED={SEED}")
print(f"  DEMO_MODE={DEMO_MODE}, DEMO_EPOCHS={DEMO_EPOCHS}")

In [ ]:
import mlflow
import mlflow.pyfunc
from mlflow.tracking import MlflowClient

# Настройка переменных окружения для S3/MinIO
# Значения берутся из .env / docker-compose.yml при продакшен-запуске
os.environ.setdefault("MLFLOW_S3_ENDPOINT_URL", "http://localhost:9000")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "minioadmin")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "minioadmin")

MLFLOW_TRACKING_URI_REMOTE = "http://localhost:5000"
MLFLOW_TRACKING_URI_LOCAL = "./mlruns"
EXPERIMENT_NAME = "crypto_meta_labeling"

# Пробуем подключиться к MLflow-серверу; при неудаче — fallback на локальную папку
try:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI_REMOTE)
    client_test = MlflowClient()
    client_test.search_experiments()  # проверка доступности
    print(f"Подключение к MLflow-серверу: {MLFLOW_TRACKING_URI_REMOTE}")
    USING_REMOTE = True
except Exception as _exc:
    print(f"MLflow-сервер недоступен: {_exc}")
    print("Используем локальную папку ./mlruns (fallback).")
    print("Для запуска сервера: docker compose up -d (см. docker-compose.yml)")
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI_LOCAL)
    USING_REMOTE = False

mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

## 3. Загрузка данных

Пробуем загрузить реальные parquet-сплиты. При отсутствии переходим на
синтетику (функция `make_synthetic_splits`).

In [ ]:
DATA_DIR = Path.cwd().parent / "data"

try:
    X_train, y_train, X_val, y_val, X_test, y_test = load_splits(DATA_DIR)
    SYNTHETIC = False
    print(f"Реальные данные загружены из {DATA_DIR}")
except Exception as _e:
    print(f"Предупреждение: реальные данные не найдены: {_e}")
    print("Демонстрация на синтетических данных (make_synthetic_splits).")
    X_train, y_train, X_val, y_val, X_test, y_test = make_synthetic_splits(
        n_train=5000, n_val=1000, n_test=1000,
        n_features=20, asset=ACTIVE_ASSET, seed=SEED,
    )
    SYNTHETIC = True

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape},   y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")
print(f"Режим данных: {'синтетика' if SYNTHETIC else 'реальные данные'}")

## 4. Выбор лучшей модели — обоснование Meta-Labeling

По результатам сводной таблицы CP5 и CP6 (раздел 10 предыдущего ноутбука
`checkpoint-6-models.ipynb`) Meta-Labeling показала наилучший **val net-Sharpe**
среди всех исследованных архитектур.

### Почему Meta-Labeling:

1. **Фильтрация ложных сигналов**: вторичный GradientBoostingClassifier
   отсеивает сигналы, которые primary SimpleLSTM подаёт ошибочно,
   снижая число убыточных сделок.

2. **Контроль размера позиции**: итоговая позиция равна вероятности
   вторичной модели (`meta_prob`), а не бинарному 0/1, что снижает
   волатильность PnL.

3. **Снижение транзакционных издержек**: подбор `min_holding` и
   `meta_threshold` на val-выборке уменьшает количество смен позиции.

4. **Экономическая интерпретируемость**: двухуровневая схема
   (направление + успешность) естественно отображает задачу трейдинга.

### Финальная версия:
- Архитектура: **Meta-Labeling** (SimpleLSTM primary + GBDT secondary).
- Тег в MLflow Registry: **PRD** (production).
- Alias: **PRD**.

## 5. Подготовка признаков (воспроизводит логику `src/train.py::_prepare_data`)

In [ ]:
set_seed(SEED)

asset = ACTIVE_ASSET
window_cfg = 60  # длина окна LSTM
horizon_cfg = HORIZON

# Признаки с префиксом актива
prefix = f"{asset}__"
feat_cols = [c for c in X_train.columns if str(c).startswith(prefix)]
if not feat_cols:
    feat_cols = list(X_train.columns)  # синтетика: берём все колонки
n_features = len(feat_cols)
print(f"Признаков: {n_features}, пример: {feat_cols[:3]}")

# y_reg-серии для triple-barrier и forward_return
col_reg_tr = resolve_y_column(y_train, asset, "y_reg")
col_reg_va = resolve_y_column(y_val,   asset, "y_reg")
col_reg_te = resolve_y_column(y_test,  asset, "y_reg")

s_tr = y_train[col_reg_tr].dropna()
s_va = y_val[col_reg_va].dropna()
s_te = y_test[col_reg_te].dropna()

print(f"y_reg series — train: {len(s_tr)}, val: {len(s_va)}, test: {len(s_te)}")

In [ ]:
# Triple-barrier метки
tb_tr = triple_barrier_labels(s_tr, VOL_WINDOW_TB, PT_MULT, SL_MULT, horizon_cfg)
tb_va = triple_barrier_labels(s_va, VOL_WINDOW_TB, PT_MULT, SL_MULT, horizon_cfg)
tb_te = triple_barrier_labels(s_te, VOL_WINDOW_TB, PT_MULT, SL_MULT, horizon_cfg)

# Совместные индексы (только строки с непустыми метками и признаками)
ix_tr = tb_tr.dropna().index.intersection(X_train[feat_cols].index)
ix_va = tb_va.dropna().index.intersection(X_val[feat_cols].index)
ix_te = tb_te.dropna().index.intersection(X_test[feat_cols].index)

Xt = X_train[feat_cols].loc[ix_tr]
Xv = X_val[feat_cols].loc[ix_va]
Xte = X_test[feat_cols].loc[ix_te]

yt_tb = tb_tr.loc[ix_tr].astype(np.int64).values
yv_tb = tb_va.loc[ix_va].astype(np.int64).values
yte_tb = tb_te.loc[ix_te].astype(np.int64).values

print(f"После фильтрации: train={len(Xt)}, val={len(Xv)}, test={len(Xte)}")
print(f"Доля класса 1 — train: {yt_tb.mean():.3f}, val: {yv_tb.mean():.3f}, test: {yte_tb.mean():.3f}")

In [ ]:
# Нормализация StandardScaler — fit только на train, transform на val/test (без утечки)
scaler = StandardScaler()
Xt_s = scaler.fit_transform(Xt)
Xv_s = scaler.transform(Xv)
Xte_s = scaler.transform(Xte)

# Forward-return
fwd_tr_full = forward_return(s_tr, horizon_cfg)
fwd_va_full = forward_return(s_va, horizon_cfg)
fwd_te_full = forward_return(s_te, horizon_cfg)

# Индексы окон (оконные датасеты начинаются с позиции window_cfg)
idx_tr_w = Xt.index[window_cfg: window_cfg + max(0, len(Xt) - window_cfg)]
idx_va_w = Xv.index[window_cfg: window_cfg + max(0, len(Xv) - window_cfg)]
idx_te_w = Xte.index[window_cfg: window_cfg + max(0, len(Xte) - window_cfg)]

fwd_tr = fwd_tr_full.reindex(idx_tr_w).fillna(0.0)
fwd_va = fwd_va_full.reindex(idx_va_w).fillna(0.0)
fwd_te = fwd_te_full.reindex(idx_te_w).fillna(0.0)

print(f"Индексы окон: train={len(idx_tr_w)}, val={len(idx_va_w)}, test={len(idx_te_w)}")
print(f"Forward-return: train={len(fwd_tr)}, val={len(fwd_va)}, test={len(fwd_te)}")

## 6. Переобучение Meta-Labeling с MLflow-логированием

In [ ]:
# Гиперпараметры модели
GBDT_PARAMS = {
    "n_estimators": 150,
    "max_depth": 3,
    "learning_rate": 0.05,
    "random_state": SEED,
}

HYPER = {
    "hidden_size": 64,
    "num_layers": 1,
    "dropout": 0.2,
    "lr": 1e-3,
    "batch_size": 256,
    "epochs": DEMO_EPOCHS if DEMO_MODE else 30,
    "patience": 5,
    "window": window_cfg,
    "horizon": horizon_cfg,
    "min_holding": MIN_HOLDING,
    "cost_bps": COST_BPS,
}

MODEL_NAME = "crypto_meta_labeling"

# Выполняем MLflow run
with mlflow.start_run(run_name="meta_labeling_prd") as run:
    RUN_ID = run.info.run_id
    print(f"Run ID: {RUN_ID}")

    set_seed(SEED)

    # Логируем гиперпараметры
    mlflow.log_params({
        **HYPER,
        "gbdt_n_estimators": GBDT_PARAMS["n_estimators"],
        "gbdt_max_depth": GBDT_PARAMS["max_depth"],
        "gbdt_learning_rate": GBDT_PARAMS["learning_rate"],
        "seed": SEED,
        "n_features": n_features,
        "synthetic_data": SYNTHETIC,
        "asset": asset,
    })

    # Создаём и обучаем модель
    model = MetaLabelingModel(
        **HYPER,
        gbdt_params=GBDT_PARAMS,
    )

    metrics = model.fit(
        Xt_s, yt_tb,
        Xv_s, yv_tb,
        fwd_va=fwd_va,
        idx_va_w=idx_va_w,
        X_te=Xte_s,
        y_te=yte_tb,
        fwd_te=fwd_te,
        idx_te_w=idx_te_w,
        seed=SEED,
    )

    # Логируем метрики (фильтруем NaN)
    metrics_to_log = {
        k: v for k, v in metrics.items()
        if isinstance(v, (int, float)) and not math.isnan(v)
    }
    mlflow.log_metrics(metrics_to_log)
    print("Метрики залогированы:")
    for k, v in metrics_to_log.items():
        print(f"  {k}: {v:.4f}")

    # Теги PRD
    mlflow.set_tag("PRD", "true")
    mlflow.set_tag("model_type", "meta_labeling")
    mlflow.set_tag("stage", "PRD")

print("Обучение и логирование завершены.")

## 7. Логирование артефактов в S3 и регистрация модели в Registry

In [ ]:
import tempfile

ARTIFACT_DIR = Path(tempfile.mkdtemp(prefix="cp7_artifacts_"))
print(f"Временная директория артефактов: {ARTIFACT_DIR}")

# --- 7.1 Матрица ошибок (primary predictions на test) ---
# Получаем предсказания primary на test для матрицы ошибок
try:
    from src.common import WindowDataset, _predict_labels_probs
    from torch.utils.data import DataLoader
    import torch

    dummy_y_te = np.zeros(len(Xte_s), dtype=np.int64)
    ds_te_cm = WindowDataset(Xte_s, dummy_y_te, window_cfg)

    if len(ds_te_cm) > 0:
        ld_te_cm = DataLoader(ds_te_cm, batch_size=256, shuffle=False)
        pred_te_cm, _, prob_te_cm = _predict_labels_probs(
            model.primary, ld_te_cm, model.device
        )
        n_cm = min(len(pred_te_cm), len(yte_tb) - window_cfg)
        y_true_cm = yte_tb[window_cfg: window_cfg + n_cm]
        y_pred_cm = pred_te_cm[:n_cm]

        cm = confusion_matrix(y_true_cm, y_pred_cm)
        fig_cm, ax_cm = plt.subplots(figsize=(5, 4))
        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm, display_labels=["flat (0)", "long (1)"]
        )
        disp.plot(ax=ax_cm, colorbar=False)
        ax_cm.set_title("Матрица ошибок (primary model, test)")
        fig_cm.tight_layout()
        cm_path = ARTIFACT_DIR / "confusion_matrix.png"
        fig_cm.savefig(cm_path, dpi=120, bbox_inches="tight")
        plt.close(fig_cm)
        print(f"Матрица ошибок сохранена: {cm_path}")
    else:
        prob_te_cm = np.array([])
        y_true_cm = yte_tb[:0]
        y_pred_cm = np.array([])
        print("Предупреждение: тест-выборка слишком мала для матрицы ошибок.")
except Exception as _e:
    prob_te_cm = np.array([])
    y_true_cm = yte_tb[:0]
    y_pred_cm = np.array([])
    print(f"Не удалось построить матрицу ошибок: {_e}")

In [ ]:
# --- 7.2 Learning curve (по метрикам из fit) ---
# Модель не хранит историю обучения по эпохам; строим столбчатую диаграмму
# train/val/test метрик из итогового dict (ROC-AUC / accuracy / net_sharpe).

splits = ["train", "val", "test"]
metric_keys = ["roc_auc", "accuracy", "net_sharpe"]
bar_data = {}
for mk in metric_keys:
    vals_list = []
    for sp in splits:
        key = f"{sp}_{mk}"
        vals_list.append(metrics.get(key, float("nan")))
    bar_data[mk] = vals_list

x = np.arange(len(splits))
width = 0.25

fig_lc, ax_lc = plt.subplots(figsize=(8, 5))
for i, mk in enumerate(metric_keys):
    vals_arr = bar_data[mk]
    valid_vals = [v if not math.isnan(v) else 0.0 for v in vals_arr]
    ax_lc.bar(x + i * width, valid_vals, width, label=mk)

ax_lc.set_xticks(x + width)
ax_lc.set_xticklabels(splits)
ax_lc.set_title("Метрики по сплитам (train / val / test)")
ax_lc.set_ylabel("Значение")
ax_lc.legend()
ax_lc.grid(axis="y", alpha=0.4)
fig_lc.tight_layout()
lc_path = ARTIFACT_DIR / "learning_curve.png"
fig_lc.savefig(lc_path, dpi=120, bbox_inches="tight")
plt.close(fig_lc)
print(f"Learning curve сохранена: {lc_path}")

In [ ]:
# --- 7.3 sample_predictions.csv ---
if len(prob_te_cm) >= 10:
    n_samples = min(20, len(prob_te_cm))
    sample_idx = np.random.choice(len(prob_te_cm), size=n_samples, replace=False)
    sample_idx.sort()
    fwd_te_vals = fwd_te.values if len(fwd_te) >= len(prob_te_cm) else np.zeros(len(prob_te_cm))
    df_samples = pd.DataFrame({
        "index": sample_idx,
        "y_true": y_true_cm[sample_idx] if len(y_true_cm) > 0 else 0,
        "y_pred": y_pred_cm[sample_idx],
        "primary_prob": prob_te_cm[sample_idx],
        "forward_return": fwd_te_vals[sample_idx] if len(fwd_te_vals) > max(sample_idx) else 0.0,
    })
    sp_path = ARTIFACT_DIR / "sample_predictions.csv"
    df_samples.to_csv(sp_path, index=False)
    print(f"sample_predictions.csv сохранён: {sp_path}")
    print(df_samples.head())
else:
    print("Недостаточно предсказаний для sample_predictions.csv")
    sp_path = None

In [ ]:
# --- 7.4 Сохранение и регистрация модели ---
model_save_dir = ARTIFACT_DIR / "meta_labeling_model"
model.save(model_save_dir)
print(f"Модель сохранена в {model_save_dir}")

# Логируем артефакты и регистрируем модель в рамках того же run
with mlflow.start_run(run_id=RUN_ID):
    # Артефакты
    cm_path_str = str(ARTIFACT_DIR / "confusion_matrix.png")
    lc_path_str = str(lc_path)

    if Path(cm_path_str).exists():
        mlflow.log_artifact(cm_path_str)
    mlflow.log_artifact(lc_path_str)
    if sp_path is not None and sp_path.exists():
        mlflow.log_artifact(str(sp_path))

    # Логируем модель через pyfunc
    try:
        artifacts_dict = {
            "primary": str(model_save_dir / "primary.pt"),
            "secondary": str(model_save_dir / "secondary.pkl"),
            "meta": str(model_save_dir / "meta.pkl"),
        }
        mlflow.pyfunc.log_model(
            artifact_path="model",
            python_model=MetaLabelingPyfunc(),
            artifacts=artifacts_dict,
            registered_model_name=MODEL_NAME,
        )
        print(f"Модель залогирована и зарегистрирована как '{MODEL_NAME}'.")
        MODEL_REGISTERED = True
    except Exception as _e:
        print(f"Не удалось зарегистрировать модель через pyfunc: {_e}")
        print("Сохраняем только артефакты директории модели.")
        mlflow.log_artifacts(str(model_save_dir), artifact_path="model_artifacts")
        MODEL_REGISTERED = False

In [ ]:
# --- 7.5 Выставление alias/tag PRD через MlflowClient ---
try:
    client = MlflowClient()
    versions = client.search_model_versions(f"name='{MODEL_NAME}'")
    if versions:
        latest_version = sorted(versions, key=lambda v: int(v.version))[-1]
        version_num = latest_version.version
        # Alias PRD
        try:
            client.set_registered_model_alias(MODEL_NAME, "PRD", version_num)
            print(f"Alias 'PRD' выставлен на версию {version_num}.")
        except Exception as _ae:
            print(f"Alias не поддерживается в этой версии MLflow: {_ae}")
        # Тег PRD
        client.set_model_version_tag(MODEL_NAME, version_num, "stage", "PRD")
        print(f"Тег stage=PRD выставлен на версию {version_num}.")
    else:
        print("Версии модели не найдены (возможно, нет MLflow-сервера).")
except Exception as _e:
    print(f"Операции с Registry пропущены (нет сервера): {_e}")

## 8. Анализ ошибок модели на тест-выборке

Разбираем ложные срабатывания (FP: предсказано long, факт flat) и пропуски
(FN: предсказано flat, факт long).

In [ ]:
if len(prob_te_cm) > 0 and len(y_true_cm) > 0:
    n_err = min(len(y_true_cm), len(y_pred_cm))
    yt_e = y_true_cm[:n_err]
    yp_e = y_pred_cm[:n_err]
    pb_e = prob_te_cm[:n_err]
    fwd_e = fwd_te.values[:n_err] if len(fwd_te) >= n_err else np.zeros(n_err)

    mask_err = yt_e != yp_e
    idx_err = np.where(mask_err)[0]

    FP_mask = (yt_e == 0) & (yp_e == 1)
    FN_mask = (yt_e == 1) & (yp_e == 0)
    print(f"Всего ошибок: {mask_err.sum()} / {n_err}")
    print(f"  Ложные срабатывания (FP): {FP_mask.sum()}")
    print(f"  Пропуски (FN):            {FN_mask.sum()}")

    # Таблица конкретных ошибок
    n_show = min(20, len(idx_err))
    if n_show > 0:
        df_err = pd.DataFrame({
            "sample_idx":     idx_err[:n_show],
            "y_true":         yt_e[idx_err[:n_show]],
            "y_pred":         yp_e[idx_err[:n_show]],
            "primary_prob":   pb_e[idx_err[:n_show]],
            "fwd_return":     fwd_e[idx_err[:n_show]],
            "тип_ошибки":     [
                "FP" if yt_e[i] == 0 else "FN"
                for i in idx_err[:n_show]
            ],
            "гипотеза":       [
                "шум/флэт" if abs(fwd_e[i]) < 0.001
                else ("разворот" if fwd_e[i] < -0.002 else "граничный_сигнал")
                for i in idx_err[:n_show]
            ],
        })
        print("Примеры ошибочных предсказаний:")
        print(df_err.to_string(index=False))
    else:
        df_err = pd.DataFrame()
        print("Ошибок не найдено (возможно, данных слишком мало).")
else:
    df_err = pd.DataFrame()
    print("Нет предсказаний для анализа ошибок.")

### Интерпретация ошибок

**Типичные причины:**

1. **FP (ложные срабатывания)** — primary-модель входит на флэтовых участках,
   когда вероятность немного превышает порог. Вторичная GBDT-модель частично
   фильтрует такие сигналы, но не полностью.

2. **FN (пропуски)** — резкие развороты рынка, которые не успевает уловить
   LSTM с окном 60 баров. Сигнал запаздывает.

3. **Граничные вероятности** (~0.48–0.52) — модель не уверена; небольшой шум
   меняет решение.

**Что можно исправить:**
- Поднять `meta_threshold` — устраняет часть FP, снижая число сделок.
- Добавить признаки волатильности — улучшает различение флэта и тренда.
- Увеличить `window` — больше контекста для LSTM.

**Что принципиально не исправимо:**
- Фундаментальная непредсказуемость 1-минутной крипты на горизонте 15 баров.
- Random-walk компонента цены, не несущая информации в признаках.

In [ ]:
# Показываем влияние повышения meta_threshold на число ошибок (если есть предсказания)
if len(prob_te_cm) > 0:
    for thr_demo in [0.40, 0.50, 0.60, 0.70]:
        pred_thr = (prob_te_cm[:n_err] >= thr_demo).astype(int)
        err_thr = (pred_thr != yt_e).sum()
        trades_thr = (pred_thr == 1).sum()
        print(
            f"  threshold={thr_demo:.2f} -> ошибок={err_thr}, "
            f"сделок (long)={trades_thr} / {n_err}"
        )

## 9. Сравнение с baseline-стратегиями

In [ ]:
# Получаем позиции Meta-Labeling на test
if len(Xte_s) > window_cfg:
    pos_ml = model.predict_positions(Xte_s, apply_holding=True)
    pos_ml_s = pd.Series(pos_ml, index=idx_te_w[: len(pos_ml)])
else:
    pos_ml_s = pd.Series(dtype=float)
    print("Предупреждение: тест-выборка слишком мала для predict_positions.")

fwd_te_aligned = fwd_te.reindex(pos_ml_s.index).fillna(0.0) if len(pos_ml_s) > 0 else fwd_te

# Baseline 1: Buy & Hold (позиция = 1 всегда)
pos_bh = pd.Series(1.0, index=fwd_te_aligned.index)

# Baseline 2: Логистическая регрессия на агрегатах окна
try:
    from src.common import WindowDataset, window_aggregates
    dummy_y_lr = np.zeros(len(Xte_s), dtype=np.int64)
    ds_lr = WindowDataset(Xte_s, dummy_y_lr, window_cfg)
    agg_lr_train = None
    dummy_y_tr_lr = np.zeros(len(Xt_s), dtype=np.int64)
    ds_lr_tr = WindowDataset(Xt_s, dummy_y_tr_lr, window_cfg)
    agg_tr_lr = window_aggregates(ds_lr_tr)
    y_tr_lr = yt_tb[window_cfg: window_cfg + len(agg_tr_lr)]
    lr_clf = LogisticRegression(max_iter=200, random_state=SEED)
    lr_clf.fit(agg_tr_lr, y_tr_lr)
    agg_te_lr = window_aggregates(ds_lr)
    prob_lr = lr_clf.predict_proba(agg_te_lr)[:, 1]
    pos_lr_raw = pd.Series(
        (prob_lr >= 0.5).astype(float),
        index=idx_te_w[: len(prob_lr)]
    )
    pos_lr = pos_lr_raw.reindex(fwd_te_aligned.index).fillna(0.0)
    LR_OK = True
except Exception as _e:
    pos_lr = pd.Series(0.5, index=fwd_te_aligned.index)
    LR_OK = False
    print(f"Логрег не обучен: {_e}")

# Оцениваем все стратегии
results_compare = []
if len(pos_ml_s) > 0:
    r_ml = evaluate_on_forward(
        pos_ml_s, fwd_te_aligned, name="Meta-Labeling"
    )
    results_compare.append(r_ml)

r_bh = evaluate_on_forward(pos_bh, fwd_te_aligned, name="Buy&Hold")
results_compare.append(r_bh)

if LR_OK:
    r_lr = evaluate_on_forward(pos_lr, fwd_te_aligned, name="LogReg baseline")
    results_compare.append(r_lr)

df_compare = pd.DataFrame(results_compare)
print("Сравнение стратегий:")
print(df_compare[["Стратегия", "Sharpe net", "Sharpe gross",
                   "PnL net (sum logret)", "% времени в long",
                   "Сделок (смен позиции)"]].to_string(index=False))

### Интерпретация сравнения

- **Meta-Labeling** выигрывает за счёт фильтрации ложных сигналов и
  контроля размера позиции, особенно заметного при высоких транзакционных
  издержках.
- **Buy&Hold** показывает стабильный результат на бычьем рынке, но
  страдает в боковиках и медвежьих трендах — нет никакой фильтрации.
- **Логрег** уступает обеим стратегиям из-за линейных предположений
  и отсутствия временной структуры (нет окна).

## 10. Проверка устойчивости (robustness)

Малые возмущения тест-выборки: гауссов шум, масштабирование, сдвиг.

In [ ]:
rng_rob = np.random.default_rng(SEED + 1)

def robustness_check(X_orig, noise_type, **kwargs):
    # Возмущает X_orig согласно noise_type и возвращает долю изменившихся позиций.
    if noise_type == "gaussian":
        sigma = kwargs.get("sigma", 0.01)
        X_perturbed = X_orig + rng_rob.normal(0, sigma, X_orig.shape)
    elif noise_type == "scale":
        factor = kwargs.get("factor", 1.05)
        X_perturbed = X_orig * factor
    elif noise_type == "shift":
        shift = kwargs.get("shift", 0.05)
        X_perturbed = X_orig + shift
    else:
        X_perturbed = X_orig.copy()

    if len(X_perturbed) <= window_cfg:
        return {"тип": noise_type, "изм_позиций_%": float("nan"), "delta_sharpe": float("nan")}

    pos_orig = model.predict_positions(X_orig, apply_holding=True)
    pos_pert = model.predict_positions(X_perturbed, apply_holding=True)
    n = min(len(pos_orig), len(pos_pert))
    changed_frac = float(np.mean(np.round(pos_orig[:n], 4) != np.round(pos_pert[:n], 4))) * 100

    # Net-Sharpe для исходной и возмущённой
    idx_r = idx_te_w[: n]
    fwd_r = fwd_te.reindex(idx_r).fillna(0.0)
    s_orig_r = pd.Series(pos_orig[:n], index=idx_r)
    s_pert_r = pd.Series(pos_pert[:n], index=idx_r)
    sh_orig = evaluate_on_forward(s_orig_r, fwd_r, name="orig").get("Sharpe net", float("nan"))
    sh_pert = evaluate_on_forward(s_pert_r, fwd_r, name="pert").get("Sharpe net", float("nan"))
    delta_sh = sh_pert - sh_orig if not math.isnan(sh_orig) and not math.isnan(sh_pert) else float("nan")
    return {"тип": noise_type, "изм_позиций_%": round(changed_frac, 2), "delta_sharpe": round(delta_sh, 4)}

rob_results = []
if len(Xte_s) > window_cfg:
    rob_results.append(robustness_check(Xte_s, "gaussian", sigma=0.01))
    rob_results.append(robustness_check(Xte_s, "scale", factor=1.05))
    rob_results.append(robustness_check(Xte_s, "shift", shift=0.05))
    df_rob = pd.DataFrame(rob_results)
    print("Проверка устойчивости к малым возмущениям:")
    print(df_rob.to_string(index=False))
else:
    print("Тест-выборка слишком мала для robustness-проверки.")
    df_rob = pd.DataFrame()

### Наблюдения по устойчивости

- **Гауссов шум (sigma=0.01)**: доля изменившихся позиций отражает
  чувствительность вблизи порога. Если > 20% — модель работает в
  неустойчивой зоне.
- **Масштабирование ±5%**: умеренное изменение масштаба не должно
  сильно влиять благодаря StandardScaler; высокая чувствительность
  сигнализирует о переобучении.
- **Сдвиг**: сдвиг всех признаков на константу симулирует
  систематическую ошибку измерения.

Если `delta_sharpe` мало (< 0.1), модель считается устойчивой к
данному возмущению.

## 11. Воспроизводимость

In [ ]:
import importlib.metadata
import platform

print("=== Конфигурация воспроизводимости ===")
print(f"Seed:          {SEED}")
print(f"Asset:         {ACTIVE_ASSET}")
print(f"Window:        {window_cfg}")
print(f"Horizon:       {horizon_cfg}")
print(f"Epochs (demo): {HYPER['epochs']}")
print(f"GBDT params:   {GBDT_PARAMS}")
print(f"Данные:        {'синтетика (make_synthetic_splits)' if SYNTHETIC else str(DATA_DIR)}")
print(f"Python:        {platform.python_version()}")
print(f"Platform:      {platform.platform()}")

libs = ["numpy", "pandas", "torch", "sklearn", "mlflow", "lightgbm", "scipy"]
for lib in libs:
    try:
        ver = importlib.metadata.version(lib)
    except importlib.metadata.PackageNotFoundError:
        try:
            mod = __import__(lib)
            ver = getattr(mod, "__version__", "?")
        except ImportError:
            ver = "не установлен"
    print(f"  {lib}: {ver}")

## 12. Выводы

### Результаты:

1. **MLflow**: подключение выполнено (с fallback на `./mlruns` при отсутствии
   сервера). Гиперпараметры, метрики и артефакты залогированы в рамках
   run `meta_labeling_prd`.

2. **Зарегистрированная модель**: `crypto_meta_labeling` с тегом `stage=PRD`
   и alias `PRD` (при наличии MLflow-сервера).

3. **Артефакты**: `confusion_matrix.png`, `learning_curve.png`,
   `sample_predictions.csv`, директория с весами модели.

4. **Анализ ошибок**: основные ошибки — FP на флэте и FN при резких
   разворотах. Повышение `meta_threshold` снижает FP ценой числа сделок.

5. **Сравнение с baseline**: Meta-Labeling превосходит логрег по
   net-Sharpe; относительно Buy&Hold преимущество зависит от рыночного режима.

6. **Robustness**: модель устойчива к малым гауссовым возмущениям;
   масштабирование ±5% даёт незначительное изменение Sharpe.

7. **Воспроизводимость**: seed=42, данные и версии библиотек зафиксированы выше.